# LLM-driven sessions on Colab (dataset step 7)

Runs `qwen2.5:7b` (via Ollama, the same model as the laptop pilot) as the agent against the
**in-process** gateway and sandbox, labels every session from the events that actually
happened, and writes the samples to Google Drive. No Docker needed; shell tasks are scripted
elsewhere, so LLM sessions use the 12 non-shell tools.

* **Runtime:** GPU (T4 is enough). Menu: *Runtime > Change runtime type > T4 GPU*.
* **Resumable:** if Colab disconnects, run all cells again; finished sessions are skipped.
* **Everything is fake:** secrets, customers and attacks target the sandbox only.
* **Pin the commit** (`COMMIT` below). Samples carry a `build_id`; the dataset builder refuses
  to mix `build_id`s, so the final run must use the same commit as the final scripted capture.

In [ ]:
# ---- parameters ---------------------------------------------------------------------------
REPO_URL = "https://github.com/kdt523/Sentinel-mcp.git"
COMMIT = "main"            # pin an exact commit hash for the FINAL run
N_BENIGN, N_ATTACK = 40, 20
AGENT_MODEL = "qwen2.5:7b"
PARAPHRASE_MODEL = "qwen2.5:3b"
SYSTEM = "plain"           # "plain" (naive agent, attacks can succeed) or "hardened"
OUT_DIR = "/content/drive/MyDrive/sentinel_llm_sessions"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import subprocess

def sh(cmd: str, cwd: str = "/content/Sentinel-mcp"):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=cwd)
    print(r.stdout[-3000:], r.stderr[-1500:] if r.returncode else "")
    r.check_returncode()

sh(f"rm -rf Sentinel-mcp && git clone -q {REPO_URL} Sentinel-mcp", cwd="/content")
sh(f"git checkout -q {COMMIT} && git log --oneline -1")
sh("pip install -q uv")
sh('uv pip install --system -q -e ".[agents,tools]" aiosqlite pytest pytest-asyncio')
sh("python -c 'import sys; print(sys.version)'")

## Ollama with the agent model

In [ ]:
import os
sh("apt-get install -y -qq zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1; ollama --version")
import subprocess, time, urllib.request
env = {**os.environ, "OLLAMA_KEEP_ALIVE": "2h"}  # keep both models loaded between requests
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except OSError:
        time.sleep(1)
sh(f"ollama pull {AGENT_MODEL} && ollama pull {PARAPHRASE_MODEL}")
sh(f"ollama show {AGENT_MODEL} | head -20")
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv")

## Warm up both models and check they run on the GPU
The first request to a model loads it, which can take a minute or more. Do this once here so
the long steps below do not time out. In `ollama ps`, the PROCESSOR column must say
**100% GPU**; if it says CPU, stop (Runtime > Change runtime type > T4 GPU) and re-run from the top.

In [ ]:
import json, time, urllib.request

def ask(model: str) -> None:
    t = time.time()
    req = urllib.request.Request(
        "http://127.0.0.1:11434/api/generate",
        data=json.dumps({"model": model, "prompt": "Say hi in three words.", "stream": False}).encode(),
        headers={"Content-Type": "application/json"},
    )
    out = json.load(urllib.request.urlopen(req, timeout=900))
    print(f"{model}: {time.time() - t:.1f}s -> {out['response'][:60]!r}")

ask(PARAPHRASE_MODEL)
ask(AGENT_MODEL)
sh("ollama ps")
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv")
sh("tail -n 12 /content/ollama.log")

## Sanity check: the pipeline works here (scripted fake model, no Ollama needed)

In [ ]:
sh("python -m pytest tests/unit/test_llm_sessions.py tests/unit/test_llm_label.py tests/unit/test_paraphrase.py -q -p no:randomly 2>&1 | tail -5")

## 1. Paraphrase the benign prompts (about 270 prompts, validated, cached)
The result `data/benign/paraphrases.json` is **committed to the repo** (builds must not need a
live model). Rewrites that drop a path, table, address, id or number are rejected.

In [ ]:
sh(f"python -m data.paraphrase --model {PARAPHRASE_MODEL} --benign {N_BENIGN} --attack {N_ATTACK}")
sh(f"cp data/benign/paraphrases.json {OUT_DIR}/paraphrases.json && python -c \"import json; d=json.load(open('data/benign/paraphrases.json')); print(len(d['prompts']), 'accepted', len(d['rejected']), 'rejected'); print(list(d['prompts'].items())[:3])\"")

## 2. Run the sessions
Appends to Drive after every session. A 7B model on a T4 needs a few seconds per session.

In [ ]:
sh(f"python -m data.llm_sessions --benign {N_BENIGN} --attack {N_ATTACK} --model {AGENT_MODEL} "
   f"--system {SYSTEM} --out {OUT_DIR}/llm_samples.jsonl --log {OUT_DIR}/llm_sessions.jsonl")

## 3. Summary

In [ ]:
import json
from collections import Counter
rows = [json.loads(l) for l in open(f"{OUT_DIR}/llm_sessions.jsonl", encoding="utf-8")]
print("sessions", len(rows), " errors", sum(bool(r["error"]) for r in rows),
      " mean seconds", round(sum(r["seconds"] for r in rows) / len(rows), 1))
print("by kind", Counter(r["kind"] for r in rows))
att = [r for r in rows if r["kind"] == "attack"]
print("attack sessions where the model carried out something malicious:",
      sum(r["malicious_events"] > 0 for r in att), "of", len(att))
fam = Counter(r["family"] for r in att); hit = Counter(r["family"] for r in att if r["malicious_events"] > 0)
for f in sorted(fam, key=str):
    print(f"  {str(f):<30} {hit[f]:>3}/{fam[f]}")
print("benign sessions with a malicious event (should be ~0):",
      sum(r["malicious_events"] > 0 for r in rows if r["kind"] != "attack"))
sh(f"git rev-parse HEAD; ollama show {AGENT_MODEL} --modelfile | grep -i '^FROM' | head -2")

## 4. Bring the results back
* `paraphrases.json` -> copy to `data/benign/paraphrases.json` in the repo and **commit** it.
* `llm_samples.jsonl` and `llm_sessions.jsonl` -> `data/raw/` on the laptop (git-ignored).
They are in `OUT_DIR` on Google Drive; the cell below also offers a direct download.

In [ ]:
from google.colab import files
for name in ("paraphrases.json", "llm_sessions.jsonl"):
    files.download(f"{OUT_DIR}/{name}")